In [1]:
import os
import pathlib
import sys
import socket

hostname = socket.gethostname()
print(hostname)

abs_path = str(pathlib.Path(hostname).parent.absolute())
WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]

if "mimi" in hostname:
    DATA_DIR = "/mn/vann/franzihe/"
    # FIG_DIR = "/uio/kant/geo-metos-u1/franzihe/Documents/Figures/mrr_reflectivity/"



INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)

mimi.uio.no


In [2]:
# supress warnings
import warnings
warnings.filterwarnings('ignore')

# import packages
from imports import(glob, datetime, pd, xr, np)

In [3]:
# reload imports
%load_ext autoreload
%autoreload 2

In [4]:
# met_file = sorted(glob(os.path.join(INPUT_DATA_DIR, f'geonor/precip_data.csv')))
met_file = sorted(glob(os.path.join(INPUT_DATA_DIR, f'geonor/GeonorCorrected.csv')))

In [5]:
met_file

['/mn/vann/franzihe/input/geonor/GeonorCorrected.csv']

In [6]:
# cols = ["TIMESTAMP","Precip1_mm", "Precip2_mm", "Precip3_mm", "Precip_Average_mm"]
cols = ["test", "cumPcpFilt"]

In [7]:
# met_p = (pd.read_csv(met_file[0], 
#                      sep=';', 
#                      usecols=cols,
#                      header = 1,
#                      skiprows=[2],
#                      parse_dates=["TIMESTAMP"],
#                      index_col="TIMESTAMP",
#                      on_bad_lines='skip',
#                      dayfirst=True,
#                     #  dtype={c: "float64" for c in cols[1:]}
#                      ).rename_axis("time").sort_index())[:-1]

met_p = (pd.read_csv(met_file[0], sep=',', usecols=cols, parse_dates=['test'], index_col='test', on_bad_lines='skip', dayfirst=True, ).rename_axis("time").sort_index())[:-1]

met_p = met_p.rename(columns={"cumPcpFilt": "precipitation"})

In [8]:
diff = met_p['precipitation'].diff()#.nsmallest(2)
diff.loc[diff!=0.].nsmallest(5)

time
2023-03-16 03:48:00    0.001000
2023-03-06 07:49:00    0.001067
2023-03-04 00:33:00    0.001100
2023-03-05 08:04:00    0.001133
2023-02-21 03:19:00    0.001133
Name: precipitation, dtype: float64

In [9]:
# for date in pd.date_range(start="2023-02-15",end="2023-03-16"):
for date in pd.date_range(start="2023-02-15",end="2023-03-16"):
    # print(date.strftime("%Y-%m-%d"))
    # met_p.loc[date.strftime("%Y-%m-%d")]
    
    met_p_day = met_p.loc[date.strftime("%Y-%m-%d")]
    if len(met_p_day) == 0:
        print(f'no data to save {date}')

    else:

        
        
        
        # ds_met_p = met_p_day.to_xarray()
        # ds_met_p['precipitation'].attrs['units'] = 'mm'

        # Step 1: Get the minute-by-minute differences
        # precip_increments = ds_met_p.diff(dim="time")
        precip_increments = met_p_day.diff()

        # # Step 2: Align the new time coordinate (since `.diff()` reduces size by 1)
        # # precip_increments["time"] = ds_met_p["time"][1:]

        # # check for negative differences and set them to Nan:
        # # precip_increments = precip_increments.where(precip_increments <= 0.01, np.nan)
        # # precip_increments[precip_increments['precipitation'] < 0.01] = np.nan
        precip_increments[precip_increments['precipitation'] <= 0.001] = np.nan
        
        # # Resample to hourly and sum 
        # pp = precip_increments.groupby(precip_increments.index.hour)['precipitation'].sum()
        # pp.index = [precip_increments[::60].index]
        pp = precip_increments.copy()
        
        # get xarray dataset
        pp = pp.to_xarray()
        # pp.attrs['units'] = 'mm/h'
        pp['precipitation'].attrs['units'] = 'mm'
        pp['precipitation'].attrs['description'] = 'Snowfall corrected GeoNor data with NAF-SEG (Ross et al., 2020)'
        
        file_name = os.path.join(INPUT_DATA_DIR, f'geonor/precip_{date.strftime("%Y%m%d")}.nc')
        pp.to_netcdf(file_name)
        print(f"saved file to {file_name}")


saved file to /mn/vann/franzihe/input/geonor/precip_20230215.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230216.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230217.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230218.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230219.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230220.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230221.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230222.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230223.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230224.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230225.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230226.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230227.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230228.nc
saved file to /mn/vann/franzihe/input/geonor/precip_20230301.nc
saved file to /mn/vann/franzihe/input/ge

In [10]:
precip_increments

,precipitation
time,
2023-03-16 00:00:00,NaN
2023-03-16 00:01:00,NaN
2023-03-16 00:02:00,NaN
2023-03-16 00:03:00,NaN
2023-03-16 00:04:00,NaN
...,...
2023-03-16 23:55:00,NaN
2023-03-16 23:56:00,NaN
2023-03-16 23:57:00,NaN


In [11]:
# # Step 3: Resample to hourly and sum
# precip_hourly = precip_increments.resample(time="1H").sum('time', skipna=True, keep_attrs=True)
# precip_hourly